In [ ]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


In [ ]:
# Load the dataset
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/ML/data.csv")  # Modify according to the path where the file is stored

print(df.head())  # Check that the data loaded correctly

Perform a brief exploratory analysis

In [ ]:
df.info()

 Most columns are of type float and contain no missing values

In [ ]:
df.describe()

We observe that the scales can vary between variables

In [ ]:
df = df.drop(columns=['Unnamed: 32'])  # Remove the empty column

In [ ]:
df['diagnosis'].value_counts()

In [ ]:
# Encode the variables B and M as 0 and 1
df['diagnosis'] = df['diagnosis'].map({'B':0, 'M':1})

We observe that there are more benign diagnoses than malignant; approximately 60% of the sample is benign

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.drop('id')
small_scale = [col for col in numeric_cols if df[col].max() < 1]
large_scale = [col for col in numeric_cols if df[col].max() >= 5]

df[small_scale].boxplot(rot=90,figsize=(15,6))
plt.show()

df[large_scale].boxplot(figsize=(15,6))
plt.show()

We observe that almost all variables have outliers, especially on the upper end. Additionally, some variables show considerable variability, while others are more concentrated around their central values. This indicates that the dataset contains a wide range of values and some extreme cases that could potentially influence the analysis.

In [ ]:
corr_matrix = df.drop(columns=['id']).corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)

We observe strong correlations between some variables.

In [ ]:
# Features and target
X = df[numeric_cols]  # all numerical columns
y = df['diagnosis']   # target variable

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33, stratify=y)

In [ ]:
# Start with the Decision Tree
grid = {
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']}

DC = DecisionTreeClassifier()
gridDC = GridSearchCV(DC, grid, cv=5)
gridDC.fit(X_train, y_train)
results = pd.DataFrame(gridDC.cv_results_)

gridDC_10 = GridSearchCV(DC, grid, cv=10)
gridDC_10.fit(X_train, y_train)
results_10 = pd.DataFrame(gridDC_10.cv_results_)

In [ ]:
# Create a function to display the best model and its accuracy
def best_model(grid_cv, X_test, y_test):
    print("Best parameters:", grid_cv.best_params_)
    print("Mean CV accuracy:", grid_cv.best_score_)

    accuracy_test = grid_cv.score(X_test, y_test)
    print("Test set accuracy:", accuracy_test)
    print("-" * 40)

    return grid_cv.best_params_, grid_cv.best_score_, accuracy_test


params, cv_score, test_score = best_model(gridDC, X_test, y_test)
params, cv_score, test_score = best_model(gridDC_10, X_test, y_test)

In [ ]:
# Normalize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Logistic Regression with hyperparameter tuning
grid_RL = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear'],
    'penalty': ['l1', 'l2']}

RL = LogisticRegression(max_iter=5000)
gridRL = GridSearchCV(RL, grid_RL, cv=5)
gridRL.fit(X_train_scaled, y_train)
results_RL = pd.DataFrame(gridRL.cv_results_)

gridRL10 = GridSearchCV(RL, grid_RL, cv=10)
gridRL10.fit(X_train_scaled, y_train)
results_RL10 = pd.DataFrame(gridRL10.cv_results_)

In [ ]:
params, cv_score, test_score = best_model(gridRL, X_test_scaled, y_test)
params, cv_score, test_score = best_model(gridRL10, X_test_scaled, y_test)


In [ ]:
# K-NN with hyperparameter tuning
grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean']
}

knn = KNeighborsClassifier()
gridknn = GridSearchCV(knn, grid_knn, cv=5)
gridknn.fit(X_train_scaled, y_train)
results_knn_cv5 = pd.DataFrame(gridknn.cv_results_)

grid_knn10 = GridSearchCV(knn, grid_knn, cv=10)
grid_knn10.fit(X_train_scaled, y_train)
results_knn_cv10 = pd.DataFrame(grid_knn10.cv_results_)

In [ ]:
params, cv_score, test_score = best_model(gridknn, X_test_scaled, y_test)
params, cv_score, test_score = best_model(grid_knn10, X_test_scaled, y_test)

All models achieve very high accuracy, indicating that the data has a clearly separable structure between classes. Additionally, the differences between k=5 and k=10 are minimal, indicating that the models are not sensitive to the data split and therefore have good generalization capability.
